## Weather Project

This project is an end to end machine learning (ML) application that predicts, tomorrow's temperature and weather, based on London's historical data, between 1979 and 2021.

The dataset contains over 15,000+ observations with features; cloud cloud_cover, sunshine, global radiation, temperature, precipitation, pressure, snow depth.

This project involves data cleaning, exploratory data analysis (EDA) using interactive visualizations, feature engeneering, model training (regression / classification) and evaluation. The final models will be integrated into a fully interactive Streamlit web application.

- By Mark

---

### 1. Loading Data

In [94]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle

In [95]:
# Load dataset
df = pd.read_csv('data/london_weather.csv')

# Structure
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

Shape: (15341, 10)
Columns: Index(['date', 'cloud_cover', 'sunshine', 'global_radiation', 'max_temp',
       'mean_temp', 'min_temp', 'precipitation', 'pressure', 'snow_depth'],
      dtype='object')


### 2. Cleaning Data

In [96]:
# Missing values
df = df.dropna(how="all")
print(f"Missing values\n{df.isna().sum()}")

Missing values
date                   0
cloud_cover           19
sunshine               0
global_radiation      19
max_temp               6
mean_temp             36
min_temp               2
precipitation          6
pressure               4
snow_depth          1441
dtype: int64


In [97]:
# Fill missing values with 0
df['cloud_cover'] = df['cloud_cover'].fillna(0)
df['snow_depth'] = df['snow_depth'].fillna(0)
df['precipitation'] = df['precipitation'].fillna(0)

# drop values that cannot be filled in
df = df.dropna(subset=["min_temp", "max_temp", "global_radiation", "pressure"])
df["mean_temp"] = df["mean_temp"].fillna((df["max_temp"] + df["min_temp"]) / 2)

In [98]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

In [99]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day_of_year

In [100]:
winter = [12,1,2]
spring = [3,4,5]
summer = [6,7,8]
autumn = [9,10,11]

def get_season(month):
    if month in winter:
        return "winter"
    elif month in spring:
        return "spring"
    elif month in autumn:
        return "autumn"
    else:
        return "summer"

df["season"] = df["month"].apply(get_season)

In [101]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,season
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,winter
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,winter
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,winter
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,winter
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,winter


### 3. Exploratory Data Analysis (EDA)

In [102]:
px.scatter(df, x="date", y="mean_temp",
           trendline="ols", opacity=0.2, trendline_color_override="black",
           title="Mean Temperature Over Time (1979-2021)",
           labels={"mean_temp": "Mean Temerature (°C)", "date": "Date (year)"})

In [103]:
px.scatter(df, x="day", y="mean_temp", color="season",
           opacity=0.4, trendline_color_override="black",
           title="Mean Daily Temperature by Year", animation_frame="year",
           labels={"mean_temp": "Mean Temerature (°C)", "day": "Day of Year"})

This plot shows the long term temprature trend in London from 1979 to 2021. We can see that there are month / season based fluctuations in temprature, with summer months being warmer and winter months being colder. However, there is also a slight upward trend in temprature over the years, indicating that London is getting warmer, likley due to climate change.

In [104]:
fig = px.line(df, x="day", y="pressure", animation_frame="year", title="Animated Daily Pressure Curve by Year",
              labels={"pressure": "Pressure (Pa)", "day": "Day of Year"})
fig.show()

In [105]:
fig = px.bar(df, x="year", y="snow_depth", title="Animated Daily Snow Depth Curve by Year",
              labels={"snow_depth": "Snow Depth (mm)", "year": "Year"}, height=400)
fig.show()

In [106]:
fig = px.line(df, x="day", y="precipitation", animation_frame="year", title="Animated Daily Precipitation Curve by Year",
              labels={"value": "Value", "day": "Day of Year"})
fig.show()

---

### 4. Feature Engineering

We need to teach the model what kinds of patterns matter, just like a human weather forcaster learns

To fo this we create new features that help the model understand:
* What happened yeaterday
* What has been happening over the last few days
* What season it is
* Weather the recent weather patters are stable or changing

In [107]:
df['temp_tomorrow'] = df["mean_temp"].shift(-1)

In [108]:
# 1 -> It will rain, 0 -> It wont rain
df["tomorrow_rain"] = (df["precipitation"].shift(-1) > 0).astype(int)

In [109]:
df = pd.get_dummies(df, columns=["season"], drop_first=True)

In [110]:
df["temp_yesterday"] = df["mean_temp"].shift(1)
df["rain_yesterday"] = df["precipitation"].shift(1)
df["sunshine_yesterday"] = df["sunshine"].shift(1)
df["pressure_yesterday"] = df["pressure"].shift(1)
df["cloud_yesterday"] = df["cloud_cover"].shift(1)

df["temp_last3"] = df["mean_temp"].rolling(3).mean()
df["pressure_last3"] = df["pressure"].rolling(3).mean()
df["sun_last3"] = df["sunshine"].rolling(3).mean()
df["cloud_last3"] = df["cloud_cover"].rolling(3).mean()
df["rain_last3"] = df["precipitation"].rolling(3).mean()

df["temp_last5"] = df["mean_temp"].rolling(5).mean()
df["pressure_last5"] = df["pressure"].rolling(5).mean()
df["sun_last5"] = df["sunshine"].rolling(5).mean()
df["cloud_last5"] = df["cloud_cover"].rolling(5).mean()
df["rain_last5"] = df["precipitation"].rolling(5).mean()

In [111]:
df = df.iloc[4:-1]

In [112]:
df.to_csv('data/features.csv', index=False)
print(f"Saved {len(df)} rows to data/features.csv")

Saved 15306 rows to data/features.csv


In [113]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,...,temp_last3,pressure_last3,sun_last3,cloud_last3,rain_last3,temp_last5,pressure_last5,sun_last5,cloud_last5,rain_last5
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,...,-2.066667,101713.333333,0.666667,6.333333,0.000000,-2.58,101914.0,2.14,5.4,0.08
5,1979-01-06,5.0,3.8,39.0,8.3,-0.5,-6.6,0.7,102780.0,1.0,...,-1.300000,101956.666667,1.933333,6.333333,0.233333,-1.86,102090.0,1.50,6.0,0.14
6,1979-01-07,8.0,0.0,13.0,8.5,1.5,-5.3,5.2,102520.0,0.0,...,0.066667,102516.666667,1.933333,6.333333,1.966667,-1.04,102088.0,1.16,6.4,1.18
7,1979-01-08,8.0,0.1,15.0,5.8,6.9,5.3,0.8,101870.0,0.0,...,2.633333,102390.000000,1.300000,7.000000,2.233333,0.90,102052.0,1.18,7.0,1.34
8,1979-01-09,4.0,5.8,50.0,5.2,3.7,1.6,7.2,101170.0,0.0,...,4.033333,101853.333333,1.966667,6.666667,4.400000,2.16,102118.0,2.34,6.2,2.78


#### Decision Tree for Feature Importance

Terms: 
* Node: The box with the question
* Leaf: The final box with a prediction
* Root: The first question
* Split: The act of diving the data based on a question
* Feature: The question asked at each node

In [114]:
df = df.drop(columns=['date', 'tomorrow_rain', 'day', 'month'])

In [115]:
# 1. Splitting the data in 

# Features
X = df.drop(columns=['temp_tomorrow'])
# Targets
y = df['temp_tomorrow']

# Splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [116]:
X_train.shape, X_test.shape

((12244, 28), (3062, 28))

In [117]:
# 2. Model fitting
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_l

In [118]:
# 3. Prediction
y_test_pred = tree.predict(X_test)

In [119]:
y_test_pred

array([12.2, 19. ,  6.8, ..., 16.6,  2. , 17.1], shape=(3062,))

In [120]:
df_test_plot = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_test_pred
})

df_test_plot.head()

,Actual,Predicted
0,13.7,12.2
1,18.0,19.0
2,7.1,6.8
3,16.6,16.0
4,9.6,9.6


In [121]:
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
importance = importance.reset_index(name="importance")

importance.T

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
index,max_temp,mean_temp,cloud_cover,min_temp,global_radiation,sunshine,temp_last5,year,pressure,temp_yesterday,...,pressure_last3,pressure_yesterday,rain_last3,cloud_yesterday,rain_yesterday,cloud_last3,season_spring,season_winter,season_summer,snow_depth
importance,0.931183,0.027877,0.004474,0.003459,0.002923,0.002645,0.002264,0.002163,0.002038,0.001849,...,0.001255,0.001212,0.000989,0.000934,0.000878,0.000683,0.000455,0.000184,0.000096,0.000038


In [122]:
fig_importance = px.bar(importance, x='index', y='importance', title='Feature Importance', log_y=True)
fig_importance.show()

### 5. Regression Models and Evaluation
We'll start with a simple linear regression model as a baseline.

In [124]:
final_features = ['max_temp', 'mean_temp', 'cloud_cover', 'min_temp', 'temp_last5', 'year', 'pressure']

In [125]:
X = df.drop(columns=['temp_tomorrow'])
X = X[final_features]

y = df['temp_tomorrow']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [126]:
model_reg = LinearRegression()
model_reg.fit(X_train, y_train)

y_pred = model_reg.predict(X_test)

In [127]:
mae = mean_absolute_error(y_test, y_pred) # (real, predicted)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.8526989972271329
RMSE: 1.103928821590903
R²: 0.9615072339328307


* MAE: On average the model's temeprature predictions are off by less than 1C.
* RMSE: The models typical error is around 1C meaning the model rarely makes large mistakes.
* R2: The model explains about 96% of the variance of tomorrows temperature based on today's weather.

In [128]:
df_eval = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

In [129]:
scatter = px.scatter(df_eval, x='Actual', y='Predicted', title="Actual vs Predicted Temperature (°C)", trendline="ols", opacity=0.7)
scatter.show()

### 6. Saving and Exporting Models

In [130]:
# Saving the model
with open("models/temperature_model.pkl", "wb") as f:
    pickle.dump(model_reg, f)

with open("models/temperature_features.pkl", "wb") as f:
    pickle.dump(final_features, f)

In [131]:
# Loading the model
with open("models/temperature_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("models/temperature_features.pkl", "rb") as f:
    feature_list = pickle.load(f)

In [132]:
print(feature_list)

['max_temp', 'mean_temp', 'cloud_cover', 'min_temp', 'temp_last5', 'year', 'pressure']


In [133]:
# Trying out out model
# Filling data from weather
user_data = {
    "max_temp": 11,         # °C, yesterday max in London
    "min_temp": 5,          # °C, yesterday min in London
    "cloud_cover": 94,      # % broken/overcast clouds around midday (approximate from conditions)
    "pressure": 1000,       # mbar (approximate from Timeanddate historic hourly data)
    "year": 2026,           # current year
    "last1": 8,             # mean temp 1 day before yesterday (approx estimate)
    "last2": 7,             # mean temp 2 days before yesterday
    "last3": 8,             # etc.
    "last4": 9,
    "last5": 8
}

In [134]:
# Mean temperature (°C)
mean_temp = (user_data["max_temp"] + user_data["min_temp"]) / 2

# 5-day rolling mean temperature (°C)
temp_last5 = np.mean([
    user_data["last1"],
    user_data["last2"],
    user_data["last3"],
    user_data["last4"],
    user_data["last5"]
])

# Pressure: mbar → Pa
pressure_pa = user_data["pressure"] * 100
# Cloud: % -> Okat
cloud_oktas = round(user_data["cloud_cover"] / 100 * 8)

input_row = {
    "max_temp": user_data["max_temp"],
    "mean_temp": mean_temp,
    "cloud_cover": cloud_oktas,
    "min_temp": user_data["min_temp"],
    "temp_last5": temp_last5,
    "year": user_data["year"],
    "pressure": pressure_pa
}

In [135]:
X_input = pd.DataFrame([input_row])

In [136]:
prediction = model.predict(X_input)[0]
print(f"Predicted temperature tomorrow: {prediction:.2f} °C")

Predicted temperature tomorrow: 8.58 °C
